In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os


In [2]:
import sys
sys.path.append("/home/z5297792/ESP_zonodo") 
from functions import doppio, out_core_param_fit

from clim_functions import doppio_pipeliner, find_directional_radii


In [3]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)

lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))

def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

i_mid, j_mid = lon_rho.shape[0] // 2, lon_rho.shape[1] // 2

dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])

x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [4]:
def rotate_uv(u, v, angle):
    u = np.where(np.abs(u) > 1e30, np.nan, u).astype(float)
    v = np.where(np.abs(v) > 1e30, np.nan, v).astype(float)
    u_rot = v * np.sin(angle) + u * np.cos(angle)
    v_rot = v * np.cos(angle) - u * np.sin(angle)

    return u_rot, v_rot


In [5]:
path = '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/df_nenc_SEACOFS_26yr.pkl'
df_nenc = pd.read_pickle(path)
df_nenc


,Day,nxc,nyc,Cyc,nic,njc,fnumber
0,1462,830.0,1515.0,AE,245,307,1461
1,1462,358.0,1408.0,AE,136,285,1461
2,1462,928.0,1356.0,CE,261,274,1461
3,1462,506.0,1354.0,CE,179,274,1461
4,1462,754.0,1285.0,AE,231,260,1461
...,...,...,...,...,...,...,...
414944,10650,349.0,158.0,AE,133,32,10641
414945,10650,973.0,126.0,CE,268,26,10641
414946,10650,805.0,95.0,AE,240,19,10641
414947,10650,157.0,34.0,CE,57,7,10641


Apply DOPPIO to Nencioli dataset. Apply it 1 day at a time, constantly saving the newly updated df. 

In [6]:
def _bad_doppio_row(row, fnumber):
    return {
        'Day': row.Day,
        'fnumber': fnumber,
        'nxc': getattr(row, 'nxc', np.nan),
        'nyc': getattr(row, 'nyc', np.nan),
        'nCyc': getattr(row, 'Cyc', np.nan),
        'xc': np.nan,
        'yc': np.nan,
        'w': np.nan,
        'Q': np.nan,
        'Omega0': np.nan,
        'Omega': np.nan,
        'Rc': np.nan,
        'psi0': np.nan,
        'R': np.nan
    }

def doppio_from_df_nenc(
    df_nenc, fnumber, X_grid, Y_grid, angle,
    nc_path='/srv/scratch/z3533156/26year_BRAN2020',
    r=30.0
):

    df_file = df_nenc.loc[df_nenc['fnumber'].eq(fnumber)]

    if df_file.empty:
        raise ValueError(f'No entries for fnumber {fnumber}')

    fname_nc = f'{nc_path}/outer_avg_{int(fnumber):05}.nc'
    print(f'Processing {int(fnumber):05}')

    out = []

    Xf = X_grid.ravel()
    Yf = Y_grid.ravel()

    with nc.Dataset(fname_nc) as ds:

        ocean_time = ds['ocean_time'][:] / 86400
        time_lookup = {int(d): i for i, d in enumerate(ocean_time)}

        for day, df_day in df_file.groupby('Day', sort=False):

            t = time_lookup.get(int(day))

            if t is None:
                for row in df_day.itertuples():
                    out.append(_bad_doppio_row(row, fnumber))
                continue

            ut_ = ds['u_eastward'][t, -1, :, :].T
            vt_ = ds['v_northward'][t, -1, :, :].T

            ut, vt = rotate_uv(ut_, vt_, angle)

            Uf = ut.ravel()
            Vf = vt.ravel()

            for row in df_day.itertuples():

                try:
                    x1, y1, u1, v1, x2, y2, u2, v2 = doppio_pipeliner(
                        row.nic, row.njc, ut, vt, X_grid, Y_grid, r=r
                    )

                    if any(np.all(np.isnan(a)) for a in [x1, y1, u1, v1, x2, y2, u2, v2]):
                        raise ValueError('bad transect')

                    xc, yc, w, Q, Omega0 = doppio(x1, y1, u1, v1, x2, y2, u2, v2)

                    radii = find_directional_radii(ut, vt, X_grid, Y_grid, xc, yc, Q)
                    R = np.nanmean([radii['up'], radii['right'], radii['down'], radii['left']])

                    dx = Xf - xc
                    dy = Yf - yc

                    rho2 = (
                        Q[0, 0] * dx**2
                        + 2 * Q[1, 0] * dx * dy
                        + Q[1, 1] * dy**2
                    )

                    rho2[rho2 < 0] = np.nan
                    rho_search = np.sqrt(rho2)

                    rho_lim = max(min(R * 1.75, 200), 30)
                    mask_outer = rho_search < rho_lim

                    Rc, psi0, Omega = out_core_param_fit(
                        Xf[mask_outer],
                        Yf[mask_outer],
                        Uf[mask_outer],
                        Vf[mask_outer],
                        xc, yc, Q,
                        Omega0=Omega0
                    )

                    out.append({
                        'Day': row.Day,
                        'fnumber': fnumber,
                        'nxc': row.nxc,
                        'nyc': row.nyc,
                        'nCyc': row.Cyc,
                        'xc': xc,
                        'yc': yc,
                        'w': w,
                        'Q': Q,
                        'Omega0': Omega0,
                        'Omega': Omega,
                        'Rc': Rc,
                        'psi0': psi0,
                        'R': R
                    })

                except Exception:
                    out.append(_bad_doppio_row(row, fnumber))

    return pd.DataFrame(out)


In [ ]:
save_file = '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/df_DOPPIO_SEACOFS_26yr.pkl'

fnumbers = np.sort(df_nenc.fnumber.dropna().unique())

# --- load existing ---
if os.path.exists(save_file):
    df_DOPPIO_SEACOFS_26yr = pd.read_pickle(save_file)

    last_fnumber = df_DOPPIO_SEACOFS_26yr['fnumber'].max() - 30
    print(f'Resuming from fnumber > {last_fnumber}')

    fnumbers = [f for f in fnumbers if f > last_fnumber]

else:
    df_DOPPIO_SEACOFS_26yr = pd.DataFrame()

# --- main loop ---
for fnumber in fnumbers:

    df_all = doppio_from_df_nenc(
        df_nenc,
        fnumber=fnumber,
        X_grid=X_grid,
        Y_grid=Y_grid,
        angle=angle
    )

    if len(df_all) == 0:
        continue

    df_all['fnumber'] = fnumber

    df_DOPPIO_SEACOFS_26yr = pd.concat(
        [df_DOPPIO_SEACOFS_26yr, df_all],
        ignore_index=True
    )

    df_DOPPIO_SEACOFS_26yr.to_pickle(save_file)

    print(f'Processed {int(fnumber):05}')



Resuming from fnumber > 2481
Processing 02511
Processed 02511
Processing 02541
Processed 02541
Processing 02571
Processed 02571
Processing 02601
Processed 02601
Processing 02631
Processed 02631
Processing 02661
Processed 02661
Processing 02691
Processed 02691
Processing 02721
Processed 02721
Processing 02751
Processed 02751
Processing 02781
Processed 02781
Processing 02811
Processed 02811
Processing 02841
Processed 02841
Processing 02871
Processed 02871
Processing 02901
Processed 02901
Processing 02931
Processed 02931
Processing 02961
Processed 02961
Processing 02991
Processed 02991
Processing 03021
Processed 03021
Processing 03051


/home/z5297792/ESP_zonodo/functions.py:523: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(


Processed 03051
Processing 03081
Processed 03081
Processing 03111
Processed 03111
Processing 03141


Dont forget that w and Omega need to be scaled from km to m